### 1.Importing Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import LabelEncoder # to label Categorical features
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import pickle #used to save Python objects into files and load them back later.
from imblearn.over_sampling import SMOTE  # SMOTE is a technique used to fix imbalanced datasets
# SMOTE = Synthetic Minority Oversampling Technique

### 2.Data Loading and Understanding

In [ ]:
#loading CSV file into Dataframe
df=pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.shape # Gives Examples,features
pd.set_option("display.max_columns",None)  #Makes sure that all columns are printed and not truncated
df.head(5) # Gives first 2 rows



In [ ]:
df.info()

In [ ]:
#Drooping customer_ID feature
df=df.drop(columns=["customerID"])
df.head()

In [ ]:
# Printing the unique values of a single feature

print(df["gender"].unique())


In [ ]:
# Printing the unique values in all the columns
for col in df.columns:
    print(col,df[col].unique())
    print("--"*50)

In [ ]:
# As u can see that the values in total_charges are in char format, we have to 
#convert them into floating point using astype(float)

df["TotalCharges"]=df["TotalCharges"].astype(float)

# DONT RUN THIS CELL :- The above code is only applicable when all the strings have a
# value in them,   ' '  cannot be converted into floating point space

In [ ]:
# printing the examples that have space in the values of TotalCharges

df[df["TotalCharges"]==" "]



In [ ]:
# Total 11 rows have no values in TotalCharges column
len(df[df["TotalCharges"]==" "])

#as tenure is zero for all these rows hence no TotalCharges is paid by them

In [ ]:
#replacing the empty scapes with '0.0'

df["TotalCharges"]=df["TotalCharges"].replace({" ":'0.0'})

In [ ]:
# Now convert the entire TotalCharges row into float

df["TotalCharges"]=df["TotalCharges"].astype(float)

## Checking Distribution on Target Values

In [ ]:
print(df["Churn"].value_counts())

### Insights 

1. Customer ID removed as it is not required for modelling
2. No missing values in the dataset
3. Converted the datatype to float of values of feature TotalCharges
4. Class imbalance identified in the target

# EDA (Exploratory Data Analysis)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df.describe()  # Gives Descriptive statistical aspects of the dataset
               # Works only on the numerical datatype

### UNDERSTANDING THE DISTRIBUTION OF THE NUMERICAL FEATURES

1. Creating a function that takes in the df and feature name, and plots the graph of that particular feature

In [ ]:
def plot_histogram(df,column_name):
    plt.figure(figsize=(5,3))
    sns.histplot(df[column_name], kde=True)
    plt.title(f"Distribution of {column_name}")

    # Calculate the mean and median values for the columns
    col_mean=df[column_name].mean()
    col_median=df[column_name].median()

    # add vertical lines for mean and median
    plt.axvline(col_mean,color="red",  linestyle="--", label="Mean")
    plt.axvline(col_median,color="green", linestyle="-", label="Median")

    plt.legend()
    plt.show()
    

    

In [ ]:
plot_histogram(df,"tenure")

In [ ]:
#U can keep on passing different features in the function

plot_histogram(df,"SeniorCitizen")

In [ ]:
plot_histogram(df,"MonthlyCharges")

In [ ]:
plot_histogram(df,"TotalCharges")

As u can see that most of the features aren't distributed normally, hence if we are going to use models OTHER than the TREE based models(desiontree, randomforest,Xgboost)

### Box plot for numerical features

In [ ]:
def plot_boxplot(df,column_name):

    plt.figure(figsize=(5,3))
    sns.boxplot(y=df[column_name])
    plt.title(f"Box_plot of {column_name}")
    plt.ylabel(column_name)
    plt.show

In [ ]:
plot_boxplot(df,"tenure")

# as u can see no outliers

In [ ]:
plot_boxplot(df,"MonthlyCharges")

# as u can see no outliers

In [ ]:
plot_boxplot(df,"TotalCharges")

# as u can see no outliers

To understand relationship between multiple numerical features we use correlation heat map 

## Correlation Heat Map

In [ ]:
#Correlation matrix - heatmap
plt.figure(figsize=(8,4))
sns.heatmap(df[["tenure","MonthlyCharges","TotalCharges"]].corr(),annot=True, cmap="coolwarm",fmt=".2f")  # Just passing the numerical features

Ignore diagonal 1's

If two columns are highly correlative(ie around 1) then we can drop one column,
we will not drop for NOW

## CATEGORICAL FEATURES ANALYSIS

In [ ]:
df.columns

In [ ]:
df.info()

For categorical features the usual plot that we would draw is the count plot

### Countplot for categorical Columns

In [ ]:
object_cols=df.select_dtypes(include="object").columns.to_list()
object_cols=["SeniorCitizen"]+object_cols  # We will also add the seniorcitizen feature into categorical features, we are doing this extra step as the SC feature int datatype
object_cols
# Putting all the categorical features into a list

In [ ]:
for col in object_cols:
    plt.figure(figsize=(5,3))
    sns.countplot(x=df[col])
    plt.title(f"Count plot of {col}")
    plt.show()

imbalance in target column is an issue

4. DATA PREPROCESSING

Label Encoding of target column


In [ ]:
df.head()

In [ ]:
df["Churn"]=df["Churn"].replace({"Yes":1,"No":0})
# Changing Yes and No to 0 and 1

In [ ]:
df.head()

Label Encoding Of categorical Features

In [ ]:
# Identifying Columns with object data type
object_columns=df.select_dtypes(include="object").columns.to_list()